# Berry Picker Assistant – Yield forecast model
**Team Picky · ML team · TIES4570 CSDP**

**Inputs** (outputs of our other notebooks, attached as Input):
- `weather_features_region_year.csv` – FMI weather features per region and year (BPA Weather Features)
- `marsi_labels.csv` – kg bought from pickers per area, berry and year (BPA Harvest Labels)

**Target – year index:** `log(kg this year / median kg for that area & berry)` → *how good was this season here compared with normal* (removes area-size effects).
**Model:** several candidates per berry (gradient boosting, ridge, a frost rule) – the one that beats the 'normal year' baseline best is shipped; if none beats it, the app gets the baseline (index 1.0) and relies on site potential + ripeness + user feedback. **Test:** leave-one-year-out (the model never sees the year it predicts) vs. a 'normal year' baseline.

⚠️ Honest limits: ~13 years × 4 areas per berry is a small dataset, and commercial purchases also depend on how many pickers came (e.g. Covid 2020). The app's own feedback will correct the score later.

In [ ]:
import glob, json, numpy as np, pandas as pd
W = pd.read_csv(glob.glob("/kaggle/input/**/weather_features_region_year.csv", recursive=True)[0])
L = pd.read_csv(glob.glob("/kaggle/input/**/marsi_labels.csv", recursive=True)[0])
print(W.shape, L.shape); print(sorted(W.region.unique())); print(L.groupby(["berry"]).year.agg(["min","max","count"]))

## 1. Map weather regions to the 4 Marsi areas and average the features

In [ ]:
AREA_OF_REGION = {
    "Lappi-south": "Lappi", "Lappi-north": "Lappi",
    "Pohjois-Pohjanmaa": "Oulu-Kainuu", "Kainuu": "Oulu-Kainuu",
    "Pohjois-Karjala": "Ita-Suomi", "Pohjois-Savo": "Ita-Suomi", "Etela-Savo": "Ita-Suomi",
}   # every other region -> Lansi-Suomi (west + south)
W["area"] = W.region.map(AREA_OF_REGION).fillna("Lansi-Suomi")
FEATS = ["gdd5_apr_aug", "gdd5_may_jun", "frost_days_flower", "min_temp_flower", "warm_days_jun",
         "rain_may_jun", "rain_jul", "rain_aug", "dry_days_jul", "tmean_jul", "snow_max_winter",
         "gdd5_apr_aug_prev", "rain_jul_prev", "rain_aug_prev"]
FEATS = [f for f in FEATS if f in W.columns]
WA = W.groupby(["area", "year"])[FEATS].mean().reset_index()
print(WA.shape)

## 2. Labels → year index

In [ ]:
lab = L[(L.area != "Koko maa") & (L.berry.isin(["bilberry", "lingonberry"])) & (L.total_kg > 0)].copy()
lab["median_kg"] = lab.groupby(["area", "berry"]).total_kg.transform("median")
lab["year_index"] = np.log(lab.total_kg / lab.median_kg)
D = lab.merge(WA, on=["area", "year"], how="inner")
AREAS = sorted(D.area.unique())
for a in AREAS: D[f"area_{a}"] = (D.area == a).astype(int)
X_COLS = FEATS + [f"area_{a}" for a in AREAS]
print(D.shape); D.groupby("berry").size()

## 3. Train candidates + leave-one-year-out test → pick the winner per berry

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from scipy.stats import spearmanr

SMALL = ["frost_days_flower", "min_temp_flower", "warm_days_jun", "rain_may_jun", "rain_jul", "gdd5_apr_aug", "snow_max_winter"]
SMALL = [f for f in SMALL if f in FEATS]
gbr   = lambda: GradientBoostingRegressor(n_estimators=150, max_depth=2, learning_rate=0.05, subsample=0.8, random_state=42)
ridge = lambda: make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-1, 3, 20)))

class FrostRule:   # no learning: -30% after >=2 frost nights during flowering (literature: frost kills flowers)
    cols = ["frost_days_flower"]
    def fit(self, X, y): return self
    def predict(self, X): return np.where(X["frost_days_flower"].values >= 2, np.log(0.7), 0.0)

class Baseline:    # "normal year" everywhere
    def fit(self, X, y): return self
    def predict(self, X): return np.zeros(len(X))

CANDIDATES = {"baseline (normal year)": (Baseline, SMALL), "frost rule": (FrostRule, ["frost_days_flower"]),
              "ridge 7 features": (ridge, SMALL), "gradient boosting all features": (gbr, X_COLS)}

def loyo(d, cols, fn):
    X, y, p = d[cols].fillna(d[cols].median()), d.year_index.values, np.zeros(len(d))
    for yr in sorted(d.year.unique()):
        tr, te = (d.year != yr).values, (d.year == yr).values
        p[te] = fn().fit(X[tr], y[tr]).predict(X[te])
    return p

rows, results, models, chosen = [], {}, {}, {}
for berry in ["bilberry", "lingonberry"]:
    d = D[D.berry == berry].reset_index(drop=True)
    for name, (fn, cols) in CANDIDATES.items():
        p = loyo(d, cols, fn)
        rho = spearmanr(d.year_index, p).correlation if np.std(p) > 0 else 0.0
        rows.append({"berry": berry, "candidate": name, "MAE": round(mean_absolute_error(d.year_index, p), 3),
                     "rank_corr": round(float(rho), 3)})
    T = pd.DataFrame([r for r in rows if r["berry"] == berry]).sort_values("MAE")
    base = T[T.candidate == "baseline (normal year)"].MAE.iloc[0]
    win = T.iloc[0]
    if win.candidate != "baseline (normal year)" and win.MAE < base * 0.97:   # must beat baseline by >=3%
        chosen[berry] = win.candidate
    else:
        chosen[berry] = "baseline (normal year)"
    fn, cols = CANDIDATES[chosen[berry]]
    kind = {"baseline (normal year)": "baseline", "frost rule": "frost_rule"}.get(chosen[berry], "sklearn")
    models[berry] = {"kind": kind, "cols": cols,   # only real sklearn models are saved -> file loads anywhere
                     "model": fn().fit(d[cols].fillna(d[cols].median()), d.year_index) if kind == "sklearn" else None}
    results[berry] = {"rows": int(len(d)), "years": int(d.year.nunique()), "MAE_baseline": float(base),
                      "chosen": chosen[berry], "MAE_chosen": float(T[T.candidate == chosen[berry]].MAE.iloc[0]),
                      "all_candidates": T.to_dict("records")}
    print(f"\n=== {berry} ({len(d)} rows, {d.year.nunique()} years) ===")
    print(T.to_string(index=False))
    print("SHIPPED:", chosen[berry])

## 3b. Info only: does weather explain which *area* does better in a given year? (removes nationwide picker effects)

In [ ]:
# Experiment: which target/model generalises to unseen years?

def loyo(d, cols, target, model_fn):
    X, y = d[cols].fillna(d[cols].median()), d[target]
    p = np.zeros(len(d))
    for yr in sorted(d.year.unique()):
        tr, te = (d.year != yr).values, (d.year == yr).values
        p[te] = model_fn().fit(X[tr], y[tr]).predict(X[te])
    mae, base = mean_absolute_error(y, p), mean_absolute_error(y, np.zeros(len(y)))
    return round(mae, 3), round(base, 3), round(100 * (1 - mae / base), 1), round(float(spearmanr(y, p).correlation), 3)

# relative target: area's share of that year's total, vs its own typical share -> removes nationwide picker effects
D["share"] = D.total_kg / D.groupby(["year", "berry"]).total_kg.transform("sum")
D["rel_index"] = np.log(D.share / D.groupby(["area", "berry"]).share.transform("median"))
for f in FEATS:   # weather relative to the other areas in the same year
    D[f + "_rel"] = D[f] - D.groupby(["year", "berry"])[f].transform("mean")
SMALL_REL = [f + "_rel" for f in SMALL]

ridge = lambda: make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-1, 3, 20)))
rows = []
for berry in ["bilberry", "lingonberry"]:
    d = D[D.berry == berry].reset_index(drop=True)
    for name, cols, target, fn in [
                ("GBR 7 rel feats / relative index", SMALL_REL, "rel_index", gbr),
        ("Ridge 7 rel feats / relative index", SMALL_REL, "rel_index", ridge),
    ]:
        rows.append((berry, name, *loyo(d, cols, target, fn)))
E = pd.DataFrame(rows, columns=["berry", "setup", "MAE_model", "MAE_baseline", "improvement_%", "rank_corr"])
print(E.to_string(index=False))
E.to_csv("/kaggle/working/experiment_results.csv", index=False)

## 4. Package for the dev team

In [ ]:
import joblib, os, shutil
H = "/kaggle/working/yield_handoff"; os.makedirs(H, exist_ok=True)
joblib.dump({"models": models, "areas": AREAS, "chosen": chosen}, f"{H}/yield_model.joblib")
json.dump({"leave_one_year_out_test": results, "features": FEATS, "areas": AREAS,
           "area_of_weather_region": AREA_OF_REGION | {"(all other regions)": "Lansi-Suomi"}},
          open(f"{H}/metrics.json", "w"), indent=2, default=float)
D.to_csv(f"{H}/training_data.csv", index=False)

In [ ]:
%%writefile /kaggle/working/yield_handoff/predict_yield.py
"""
predict_yield.py  -  Yield forecast for Berry Picker Assistant (handoff to the dev team)

1) Year index: how good this season is in an area compared with a normal year
   (from weather; model trained on 2013-2025 Marsi harvest data + FMI weather).
2) Zone score 0-100 = year index x site potential (forest type) x ripeness timing.

Usage:
    from predict_yield import YieldModel
    ym = YieldModel("yield_model.joblib")
    idx = ym.year_index("Ita-Suomi", "bilberry", weather_features)   # 1.0 = normal year, 1.5 = 50% better
    score = ym.zone_score(idx, "bilberry", forest_type="mesic_heath", day_of_year=205)
"""
import math
import joblib
import numpy as np
import pandas as pd

# Typical yields of an average year by forest site type (kg/ha), from Finnish berry research (Luke):
# bilberry is best in mesic heath (MT) forests, lingonberry in sub-xeric (VT) pine forests and young stands.
SITE_POTENTIAL = {
    "bilberry":    {"herb_rich": 5,  "mesic_heath": 25, "sub_xeric": 15, "xeric": 5,  "peatland": 10, "clearcut": 2},
    "lingonberry": {"herb_rich": 2,  "mesic_heath": 10, "sub_xeric": 30, "xeric": 20, "peatland": 10, "clearcut": 25},
    "cloudberry":  {"herb_rich": 0,  "mesic_heath": 0,  "sub_xeric": 0,  "xeric": 0,  "peatland": 15, "clearcut": 0},
}
# Typical ripening windows in southern/central Finland (day of year); shift later further north
RIPENING = {"bilberry": (190, 240), "lingonberry": (225, 280), "cloudberry": (185, 215)}


class YieldModel:
    def __init__(self, path="yield_model.joblib"):
        b = joblib.load(path)
        self.models, self.areas, self.chosen = b["models"], b["areas"], b["chosen"]

    def year_index(self, area, berry, weather):
        """1.0 = normal year. Cloudberry (no model) always returns 1.0. Result clipped to 0.5-1.5."""
        if berry not in self.models:
            return 1.0
        m = self.models[berry]
        if m["kind"] == "baseline":
            return 1.0
        if m["kind"] == "frost_rule":   # -30% after >=2 frost nights during flowering
            return 0.7 if (weather.get("frost_days_flower") or 0) >= 2 else 1.0
        row = {**{f: weather.get(f, np.nan) for f in m["cols"] if not f.startswith("area_")},
               **{f"area_{a}": int(a == area) for a in self.areas}}
        X = pd.DataFrame([row])[m["cols"]].astype(float).fillna(0)
        return float(min(1.5, max(0.5, math.exp(m["model"].predict(X)[0]))))

    def zone_score(self, year_index, berry, forest_type="mesic_heath", day_of_year=None):
        best = max(SITE_POTENTIAL[berry].values()) or 1
        site = SITE_POTENTIAL[berry].get(forest_type, 0) / best
        ripe = 1.0
        if day_of_year is not None:
            a, b = RIPENING[berry]
            ripe = 0.0 if day_of_year < a - 10 or day_of_year > b + 10 else (0.5 if day_of_year < a or day_of_year > b else 1.0)
        return round(max(0, min(100, 100 * site * ripe * min(year_index, 2.0) / 2.0 * 1.6)), 1)


In [ ]:
import sys; sys.path.insert(0, H)
from predict_yield import YieldModel
ym = YieldModel(f"{H}/yield_model.joblib")
w = WA[(WA.area == "Ita-Suomi") & (WA.year == WA.year.max())].iloc[0].to_dict()
for berry in ["bilberry", "lingonberry"]:
    idx = ym.year_index("Ita-Suomi", berry, w)
    print(berry, "year index", round(idx, 2), "| zone score (best forest type, ripe):",
          ym.zone_score(idx, berry, "mesic_heath" if berry == "bilberry" else "sub_xeric", 210 if berry == "bilberry" else 240))

In [ ]:
readme = f"""# Yield forecast – handoff for the dev team (Team Picky)

## What it does
1. **year_index(area, berry, weather)** – how good this season is in an area vs a normal year (1.0 = normal, 1.5 = +50%).
   Candidates (gradient boosting, ridge, frost rule) were tested against a 'normal year' baseline on unseen years; the winner per berry is shipped: {chosen}.\n   If the baseline won, year_index always returns 1.0 – that is the honest answer: our data could not predict seasons better than 'normal'.
2. **zone_score(year_index, berry, forest_type, day_of_year)** – 0–100 score for one map zone:
   year index × site potential of the forest type × ripeness timing.
   The app's trust-weighted picker feedback and photo berry counts should then correct this score.

## Files
| File | What |
|---|---|
| `yield_model.joblib` | shipped model per berry (see metrics.json → chosen) |
| `predict_yield.py` | `YieldModel` class |
| `metrics.json` | leave-one-year-out test results |
| `training_data.csv` | the joined training table |

## Weather features expected
{", ".join(FEATS)} (computed per area from FMI daily data, see the BPA Weather Features notebook).
Areas: {", ".join(AREAS)}.

## Accuracy (leave-one-year-out, log year index)
{json.dumps({b: {k: v for k, v in r.items() if k != "all_candidates"} for b, r in results.items()}, indent=2)}

## Limits
Small data (≈13 years × 4 areas), and commercial purchases also depend on picker numbers. Use as a prior, not as truth.
Cloudberry: no model yet (too little data) – use site potential + ripeness only.
"""
open(f"{H}/README.md", "w").write(readme)
shutil.make_archive("/kaggle/working/yield_handoff", "zip", H)
!ls -la /kaggle/working/yield_handoff